In [2]:
import pandas as pd
import sys
sys.path.append("../src")
from utils.utils import data_report

In [3]:
df_1trimestre = pd.read_csv("../data/raw/Airbnb_2025/listings_2025_03_05.csv.gz")
df_1trimestre.head(3)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,21853,https://www.airbnb.com/rooms/21853,20250305023340,2025-03-11,city scrape,Bright and airy room,We have a quiet and sunny room with a good vie...,We live in a leafy neighbourhood with plenty o...,https://a0.muscache.com/pictures/68483181/87bc...,83531,...,4.82,4.21,4.67,NaN,f,2,0,2,0,0.26
1,30320,https://www.airbnb.com/rooms/30320,20250305023340,2025-03-10,previous scrape,Great Vacational Apartments,NaN,NaN,https://a0.muscache.com/pictures/336868/f67409...,130907,...,4.78,4.90,4.69,NaN,f,3,3,0,0,0.96
2,30959,https://www.airbnb.com/rooms/30959,20250305023340,2025-03-10,previous scrape,Beautiful loft in Madrid Center,Beautiful Loft 60m2 size just in the historica...,NaN,https://a0.muscache.com/pictures/78173471/835e...,132883,...,4.63,4.88,4.25,NaN,f,1,1,0,0,0.07


In [4]:
df_1trimestre.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25288 entries, 0 to 25287
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            25288 non-null  int64  
 1   listing_url                                   25288 non-null  object 
 2   scrape_id                                     25288 non-null  int64  
 3   last_scraped                                  25288 non-null  object 
 4   source                                        25288 non-null  object 
 5   name                                          25288 non-null  object 
 6   description                                   24375 non-null  object 
 7   neighborhood_overview                         11218 non-null  object 
 8   picture_url                                   25287 non-null  object 
 9   host_id                                       25288 non-null 

### Carga de los snapshots de Inside Airbnb

Inside Airbnb publica una foto completa de los anuncios de Madrid en
fechas concretas. Trabajamos con tres de 2025: marzo, junio y septiembre.

No son periodos consecutivos: cada archivo contiene todos los anuncios
activos ese día, así que un mismo alojamiento aparece en los tres.

El target, `estimated_revenue_l365d`, es el ingreso estimado de los 365
días previos al scraping y solo existe para los anuncios con precio
publicado (19.274 de 25.288 en este primer snapshot).

>El target, `estimated_revenue_l365d`, es el ingreso estimado de los 365
días previos al scraping. Solo tiene valor en 19.274 de 25.288 filas,
exactamente las mismas que tienen `price` informado.

>Esa coincidencia exacta llama la atención y hay que tenerla en cuenta:
apunta a que el revenue podría no ser un dato medido, sino un valor
derivado de otras columnas del propio dataset. Habrá que comprobarlo
antes de modelar, porque de confirmarse sería una fuente de leakage.

In [5]:
df_2trimestre = pd.read_csv("../data/raw/Airbnb_2025/listings_2025_06_12.csv.gz")

In [6]:
df_2trimestre.shape

(26004, 79)

In [7]:
df_3trimestre = pd.read_csv("../data/raw/Airbnb_2025/listings_2025_09_14.csv.gz")

In [8]:
df_3trimestre.shape

(25000, 79)

In [9]:
df_alquileres_original = pd.concat([df_1trimestre, df_2trimestre, df_3trimestre])
df_alquileres_original.head(3)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,21853,https://www.airbnb.com/rooms/21853,20250305023340,2025-03-11,city scrape,Bright and airy room,We have a quiet and sunny room with a good vie...,We live in a leafy neighbourhood with plenty o...,https://a0.muscache.com/pictures/68483181/87bc...,83531,...,4.82,4.21,4.67,NaN,f,2,0,2,0,0.26
1,30320,https://www.airbnb.com/rooms/30320,20250305023340,2025-03-10,previous scrape,Great Vacational Apartments,NaN,NaN,https://a0.muscache.com/pictures/336868/f67409...,130907,...,4.78,4.90,4.69,NaN,f,3,3,0,0,0.96
2,30959,https://www.airbnb.com/rooms/30959,20250305023340,2025-03-10,previous scrape,Beautiful loft in Madrid Center,Beautiful Loft 60m2 size just in the historica...,NaN,https://a0.muscache.com/pictures/78173471/835e...,132883,...,4.63,4.88,4.25,NaN,f,1,1,0,0,0.07


In [10]:
df_alquileres_original.shape

(76292, 79)

### Unificación de los tres snapshots

La concatenación da 76.292 filas manteniendo las 79 columnas, así que
los tres archivos comparten estructura.

No son 76.292 alojamientos distintos: un anuncio activo todo el año
aparece una vez por snapshot. Usamos el `id` como índice y eliminamos
duplicados quedándonos con su versión más reciente.

In [11]:
df_alquileres_original = df_alquileres_original.set_index("id") 
df_alquileres_original.head(3)

,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
id,,,,,,,,,,,,,,,,,,,,,
21853,https://www.airbnb.com/rooms/21853,20250305023340,2025-03-11,city scrape,Bright and airy room,We have a quiet and sunny room with a good vie...,We live in a leafy neighbourhood with plenty o...,https://a0.muscache.com/pictures/68483181/87bc...,83531,https://www.airbnb.com/users/show/83531,...,4.82,4.21,4.67,NaN,f,2,0,2,0,0.26
30320,https://www.airbnb.com/rooms/30320,20250305023340,2025-03-10,previous scrape,Great Vacational Apartments,NaN,NaN,https://a0.muscache.com/pictures/336868/f67409...,130907,https://www.airbnb.com/users/show/130907,...,4.78,4.90,4.69,NaN,f,3,3,0,0,0.96
30959,https://www.airbnb.com/rooms/30959,20250305023340,2025-03-10,previous scrape,Beautiful loft in Madrid Center,Beautiful Loft 60m2 size just in the historica...,NaN,https://a0.muscache.com/pictures/78173471/835e...,132883,https://www.airbnb.com/users/show/132883,...,4.63,4.88,4.25,NaN,f,1,1,0,0,0.07


In [12]:
print(f"Duplicados en el id: {df_alquileres_original.index.duplicated().sum()}")

Duplicados en el id: 45061


In [13]:
duplicadas_id = df_alquileres_original[df_alquileres_original.index.duplicated(keep=False)]
duplicadas_id = duplicadas_id.sort_index(ascending=False)
duplicadas_id.head(2)

,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
id,,,,,,,,,,,,,,,,,,,,,
1441028226979579527,https://www.airbnb.com/rooms/1441028226979579527,20250612050748,2025-06-26,city scrape,Moderna habitacion en malasaña,The whole group will enjoy easy access to ever...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,675465242,https://www.airbnb.com/users/show/675465242,...,5.0,5.0,4.0,NaN,f,1,0,1,0,1.00
1441028226979579527,https://www.airbnb.com/rooms/1441028226979579527,20250914152907,2025-09-14,city scrape,Habitación de lujo en Plaza de España,The whole group will enjoy easy access to ever...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,675465242,https://www.airbnb.com/users/show/675465242,...,5.0,5.0,4.0,NaN,f,1,0,1,0,0.36


In [14]:
duplicadas_id["last_scraped"]

id
1441028226979579527    2025-06-26
1441028226979579527    2025-09-14
1440753543897517467    2025-06-25
1440753543897517467    2025-09-15
1440701633811148099    2025-09-15
                          ...    
30320                  2025-03-10
30320                  2025-09-15
21853                  2025-03-11
21853                  2025-09-15
21853                  2025-06-26
Name: last_scraped, Length: 70471, dtype: object

In [15]:
# Ordenamos por last_scraped porque el orden de concat no garantiza
# que la última fila de cada id sea la del snapshot más reciente

df_alquileres_original = df_alquileres_original.sort_values("last_scraped")

In [16]:
df_alquileres_original.loc[21853, "last_scraped"]

id
21853    2025-03-11
21853    2025-06-26
21853    2025-09-15
Name: last_scraped, dtype: object

In [17]:
df_alquileres_original = df_alquileres_original[~df_alquileres_original.index.duplicated(keep= "last")]
df_alquileres_original.index.duplicated().sum()

np.int64(0)

In [18]:
df_alquileres_original.shape

(31231, 78)

### Eliminación de duplicados

Nos quedamos con una única fila por anuncio, la del snapshot más
reciente. De 76.292 filas pasamos a 31.231 alojamientos distintos.

In [24]:
pd.set_option("display.max_rows", None)
data_report(df_alquileres_original).T

,DATA_TYPE,MISSINGS (%),UNIQUE_VALUES,CARDIN (%)
COL_N,,,,
listing_url,object,0.0,31231,100.0
scrape_id,int64,0.0,3,0.01
last_scraped,object,0.0,14,0.04
source,object,0.0,2,0.01
name,object,0.0,28883,92.48
description,object,3.33,24733,79.19
neighborhood_overview,object,59.41,9505,30.43
picture_url,object,0.01,30720,98.36
host_id,int64,0.0,13196,42.25


In [21]:
df_alquileres_original["scrape_id"].nunique(), df_alquileres_original["last_scraped"].nunique()

(3, 14)

In [25]:
df_alquileres_original.to_csv("../data/processed/df_alquileres_original.csv", index= True, encoding= "utf-8")